# BirdCLEF 2026 — Exploratory Data Analysis

**Competition**: [BirdCLEF 2026](https://www.kaggle.com/competitions/birdclef-2026)  
**Full analysis**: [docs/references/birdclef-2026-competition.md](../../../docs/references/birdclef-2026-competition.md)

## Scope

This notebook focuses on the **four most actionable** EDA questions for informing model design:

1. **Species Distribution & Class Balance** — per-class and per-species histograms, the long tail, collection breakdown
2. **Soundscape Label Analysis** — co-occurrence patterns, temporal patterns, species overlap with train_audio
3. **Soundscape-Only Species Deep Dive** — the 28 species with no focal recordings, training sample counts, co-occurrence
4. **Secondary Labels Analysis** — background species in focal recordings, multi-label coverage

### Future Work (not covered here)

5. **Audio Characteristics** — duration distributions, quality ratings by class/collection, iNat vs XC differences
6. **Geographic & Temporal Patterns** — site distributions, time-of-day patterns, seasonal variation

## Setup

In [ ]:
from __future__ import annotations

import ast
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 120

# Data paths
DATA_DIR = Path("/storage/project/ps-clef2026_birdclef-0/shared/data/birdclef-2026")
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

# Load core CSVs
train_df = pd.read_csv(DATA_DIR / "train.csv")
taxonomy_df = pd.read_csv(DATA_DIR / "taxonomy.csv")
ss_labels_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
submission_df = pd.read_csv(DATA_DIR / "sample_submission.csv", nrows=1)

# Build taxonomy lookup
tax_lookup = taxonomy_df.set_index("primary_label")[
    ["common_name", "class_name", "scientific_name"]
].to_dict("index")

# Submission species set
submission_species = set(submission_df.columns[1:])

print(f"Train recordings:        {len(train_df):,}")
print(f"Taxonomy species:        {len(taxonomy_df)}")
print(f"Submission species:      {len(submission_species)}")
print(f"Soundscape label rows:   {len(ss_labels_df):,}")
print(f"Train soundscape files:  {len(os.listdir(DATA_DIR / 'train_soundscapes')):,}")

---

## 1. Species Distribution & Class Balance

How are recordings distributed across taxonomic classes and individual species? How severe is the long-tail imbalance?

### 1.1 Taxonomic class breakdown

In [ ]:
# Class breakdown in train_audio
class_counts = train_df["class_name"].value_counts()
species_per_class = train_df.groupby("class_name")["primary_label"].nunique()

class_summary = pd.DataFrame(
    {
        "recordings": class_counts,
        "species": species_per_class,
        "pct_recordings": (class_counts / len(train_df) * 100).round(1),
    }
).sort_values("recordings", ascending=False)

print("=== Taxonomic class breakdown (train_audio) ===")
display(class_summary)

# Class breakdown in taxonomy (submission species)
tax_class_counts = taxonomy_df["class_name"].value_counts()
print("\n=== Submission species per class (taxonomy.csv) ===")
display(tax_class_counts.to_frame("species"))

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recordings per class
class_summary["recordings"].plot.bar(
    ax=axes[0], color=sns.color_palette("muted", len(class_summary))
)
axes[0].set_title("Focal Recordings per Taxonomic Class")
axes[0].set_ylabel("Number of recordings")
axes[0].set_xlabel("")
for i, v in enumerate(class_summary["recordings"]):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontsize=9)

# Species per class (submission vs train_audio)
comparison = pd.DataFrame(
    {
        "train_audio": species_per_class.reindex(tax_class_counts.index, fill_value=0),
        "submission (total)": tax_class_counts,
    }
)
comparison.plot.bar(ax=axes[1], color=["steelblue", "coral"])
axes[1].set_title("Species Count: train_audio vs Submission")
axes[1].set_ylabel("Number of species")
axes[1].set_xlabel("")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

### 1.2 Per-species recording distribution (long tail)

In [ ]:
# Per-species recording counts (train_audio only)
species_counts = train_df["primary_label"].value_counts().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rank-ordered distribution (log scale)
axes[0].bar(
    range(len(species_counts)), species_counts.values, width=1.0, color="steelblue", alpha=0.8
)
axes[0].set_yscale("log")
axes[0].set_xlabel("Species rank (sorted by count)")
axes[0].set_ylabel("Number of recordings (log)")
axes[0].set_title("Species Recording Distribution (train_audio)")
axes[0].axhline(y=10, color="red", linestyle="--", alpha=0.6, label="10 recordings")
axes[0].axhline(y=50, color="orange", linestyle="--", alpha=0.6, label="50 recordings")
axes[0].legend()

# Histogram of recording counts
axes[1].hist(species_counts.values, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Number of recordings")
axes[1].set_ylabel("Number of species")
axes[1].set_title("Distribution of Recording Counts per Species")
axes[1].axvline(
    x=species_counts.median(),
    color="red",
    linestyle="--",
    label=f"Median: {species_counts.median():.0f}",
)
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary stats
print(f"Mean:   {species_counts.mean():.1f}")
print(f"Median: {species_counts.median():.0f}")
print(f"Max:    {species_counts.max()} ({species_counts.idxmax()})")
print(f"Min:    {species_counts.min()} ({species_counts.idxmin()})")
print(f"\nSpecies with < 10 recordings: {(species_counts < 10).sum()}")
print(f"Species with < 50 recordings: {(species_counts < 50).sum()}")
print(f"Species with < 100 recordings: {(species_counts < 100).sum()}")

### 1.3 Collection breakdown (XC vs iNat) by class

In [ ]:
# Collection x Class crosstab
coll_class = pd.crosstab(train_df["class_name"], train_df["collection"], margins=True)
print("=== Recordings by Collection x Class ===")
display(coll_class)

# Visualize
coll_class_no_margin = coll_class.drop("All", axis=0).drop("All", axis=1)
coll_class_no_margin.plot.bar(stacked=True, figsize=(10, 5), color=["steelblue", "coral"])
plt.title("Recordings by Collection and Taxonomic Class")
plt.ylabel("Number of recordings")
plt.xlabel("")
plt.legend(title="Collection")
plt.tight_layout()
plt.show()

# Which classes are iNat-dominant vs XC-dominant?
pct_inat = (coll_class_no_margin.get("iNat", 0) / coll_class_no_margin.sum(axis=1) * 100).round(1)
print("\n=== % iNaturalist by class ===")
for cls, pct in pct_inat.items():
    print(f"  {cls}: {pct}% iNat")

---

## 2. Soundscape Label Analysis

The 66 labeled soundscape files with 1,478 expert-annotated 5s segments are new to BirdCLEF 2026. Understanding the label structure is critical for designing training on soundscape data.

### 2.1 Label density — how many species per segment?

In [ ]:
# Parse multi-label column
ss_labels_df["species_list"] = ss_labels_df["primary_label"].str.split(";")
ss_labels_df["n_species"] = ss_labels_df["species_list"].apply(len)

# Explode to one row per species-segment pair
ss_exploded = ss_labels_df.explode("species_list").rename(columns={"species_list": "species"})

print(f"Total labeled segments:     {len(ss_labels_df):,}")
print(f"Unique soundscape files:    {ss_labels_df['filename'].nunique()}")
print(f"Unique species in labels:   {ss_exploded['species'].nunique()}")
print(f"Mean species per segment:   {ss_labels_df['n_species'].mean():.1f}")
print(f"Max species per segment:    {ss_labels_df['n_species'].max()}")

# Label density histogram
fig, ax = plt.subplots(figsize=(10, 5))
ss_labels_df["n_species"].value_counts().sort_index().plot.bar(
    ax=ax, color="steelblue", edgecolor="white"
)
ax.set_xlabel("Number of species per 5s segment")
ax.set_ylabel("Number of segments")
ax.set_title("Label Density in Expert-Annotated Soundscapes")
for i, (_idx, val) in enumerate(ss_labels_df["n_species"].value_counts().sort_index().items()):
    ax.text(i, val + 5, str(val), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

### 2.2 Species frequency in soundscapes

In [ ]:
from matplotlib.patches import Patch

# Species frequency in labeled soundscapes
ss_species_counts = ss_exploded["species"].value_counts()

# Add class info
ss_species_freq = ss_species_counts.reset_index()
ss_species_freq.columns = ["species", "segments"]
ss_species_freq["class"] = ss_species_freq["species"].map(
    lambda s: tax_lookup.get(s, {}).get("class_name", "?")
)
ss_species_freq["common_name"] = ss_species_freq["species"].map(
    lambda s: tax_lookup.get(s, {}).get("common_name", "?")
)

# Color by class
class_colors = {
    "Aves": "steelblue",
    "Amphibia": "forestgreen",
    "Insecta": "orange",
    "Mammalia": "red",
    "Reptilia": "purple",
}

fig, ax = plt.subplots(figsize=(16, 8))
colors = [class_colors.get(c, "gray") for c in ss_species_freq["class"]]
ax.bar(range(len(ss_species_freq)), ss_species_freq["segments"], color=colors, width=1.0)
ax.set_xlabel("Species rank (sorted by segment count)")
ax.set_ylabel("Number of 5s segments")
ax.set_title("Species Frequency in Labeled Soundscapes (colored by taxonomic class)")

# Legend
legend_elements = [
    Patch(facecolor=v, label=k)
    for k, v in class_colors.items()
    if k in ss_species_freq["class"].values
]
ax.legend(handles=legend_elements, loc="upper right")

plt.tight_layout()
plt.show()

# Top 20 species
print("=== Top 20 species in labeled soundscapes ===")
display(ss_species_freq.head(20))

### 2.3 Species co-occurrence in soundscapes

Which species tend to appear together? This reveals ecological groupings (e.g., nocturnal frog choruses, dawn bird choruses) and informs whether MixIT separation targets are realistic.

In [ ]:
# Build co-occurrence matrix from labeled soundscape segments
ss_species_list = ss_exploded["species"].unique()
top_n = 30  # Focus on top-N most frequent species for readability
top_species = ss_species_counts.head(top_n).index.tolist()

# Binary presence matrix (segments x species)
presence = pd.DataFrame(0, index=ss_labels_df.index, columns=top_species)
for idx, row in ss_labels_df.iterrows():
    for sp in row["species_list"]:
        if sp in top_species:
            presence.at[idx, sp] = 1

# Co-occurrence matrix
cooccurrence = presence.T @ presence
np.fill_diagonal(cooccurrence.values, 0)  # Remove self-co-occurrence

# Add common names for readability
labels = [f"{s}\n({tax_lookup.get(s, {}).get('common_name', s)[:20]})" for s in top_species]
short_labels = [tax_lookup.get(s, {}).get("common_name", s)[:18] for s in top_species]

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cooccurrence,
    xticklabels=short_labels,
    yticklabels=short_labels,
    cmap="YlOrRd",
    ax=ax,
    linewidths=0.5,
    fmt="d",
    annot=False,
)
ax.set_title(f"Species Co-occurrence in Labeled Soundscapes (top {top_n})")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Top co-occurring pairs
pairs = []
for i, sp1 in enumerate(top_species):
    for j, sp2 in enumerate(top_species):
        if i < j:
            pairs.append(
                {
                    "species_1": sp1,
                    "species_2": sp2,
                    "name_1": tax_lookup.get(sp1, {}).get("common_name", sp1),
                    "name_2": tax_lookup.get(sp2, {}).get("common_name", sp2),
                    "class_1": tax_lookup.get(sp1, {}).get("class_name", "?"),
                    "class_2": tax_lookup.get(sp2, {}).get("class_name", "?"),
                    "co_occurrences": cooccurrence.at[sp1, sp2],
                }
            )

pairs_df = pd.DataFrame(pairs).sort_values("co_occurrences", ascending=False)
pairs_df["cross_class"] = pairs_df["class_1"] != pairs_df["class_2"]

print("=== Top 15 co-occurring species pairs ===")
display(
    pairs_df.head(15)[["name_1", "class_1", "name_2", "class_2", "co_occurrences", "cross_class"]]
)

print("\n=== Cross-class co-occurrences (e.g., bird + frog in same segment) ===")
cross_class = pairs_df[pairs_df["cross_class"]].head(10)
display(cross_class[["name_1", "class_1", "name_2", "class_2", "co_occurrences"]])

---

## 3. Soundscape-Only Species Deep Dive

28 species in the submission have **no focal recordings** in train_audio. They exist only in the 66 labeled soundscape files. These are the hardest species to learn — understanding exactly how much data we have for each is critical.

### 3.1 Identify soundscape-only species and their data availability

In [ ]:
# Species in train_audio vs soundscape labels vs submission
train_audio_species = set(train_df["primary_label"].unique())
ss_label_species = set(ss_exploded["species"].unique())

soundscape_only = submission_species - train_audio_species
in_both = train_audio_species & ss_label_species
train_only = train_audio_species - ss_label_species
ss_only_actual = ss_label_species - train_audio_species
no_data = submission_species - train_audio_species - ss_label_species

print("=== Species coverage ===")
print(f"  Submission species:              {len(submission_species)}")
print(f"  In train_audio:                  {len(train_audio_species)}")
print(f"  In soundscape labels:            {len(ss_label_species)}")
print(f"  In both:                         {len(in_both)}")
print(f"  Only in train_audio:             {len(train_only)}")
print(f"  Only in soundscape labels:       {len(ss_only_actual)}")
print(f"  In submission but NO data at all: {len(no_data)}")

# Detailed view of soundscape-only species
ss_only_data = []
for sp in sorted(soundscape_only):
    info = tax_lookup.get(sp, {})
    seg_count = ss_species_counts.get(sp, 0)
    # Count unique soundscape files containing this species
    file_count = (
        ss_exploded[ss_exploded["species"] == sp]["filename"].nunique() if seg_count > 0 else 0
    )
    ss_only_data.append(
        {
            "species": sp,
            "common_name": info.get("common_name", "?"),
            "class": info.get("class_name", "?"),
            "labeled_segments": seg_count,
            "soundscape_files": file_count,
            "in_soundscape_labels": sp in ss_label_species,
        }
    )

ss_only_df = pd.DataFrame(ss_only_data).sort_values("labeled_segments", ascending=False)

print(f"\n=== Soundscape-only species ({len(ss_only_df)}) ===")
display(ss_only_df)

In [ ]:
# Visualize soundscape-only species data availability
fig, ax = plt.subplots(figsize=(14, 8))

# Only species that actually appear in labels
ss_only_with_data = ss_only_df[ss_only_df["labeled_segments"] > 0].copy()
ss_only_no_data = ss_only_df[ss_only_df["labeled_segments"] == 0]

colors = [class_colors.get(c, "gray") for c in ss_only_with_data["class"]]
bars = ax.barh(range(len(ss_only_with_data)), ss_only_with_data["labeled_segments"], color=colors)

# Labels
ytick_labels = [
    f"{row['species']} ({row['common_name'][:25]})" for _, row in ss_only_with_data.iterrows()
]
ax.set_yticks(range(len(ss_only_with_data)))
ax.set_yticklabels(ytick_labels, fontsize=8)
ax.set_xlabel("Number of labeled 5s segments")
ax.set_title(
    f"Training Data for Soundscape-Only Species ({len(ss_only_with_data)} with data, {len(ss_only_no_data)} with none)"
)
ax.invert_yaxis()

# Add count labels
for i, v in enumerate(ss_only_with_data["labeled_segments"]):
    ax.text(v + 2, i, str(v), va="center", fontsize=8)

legend_elements = [
    Patch(facecolor=v, label=k)
    for k, v in class_colors.items()
    if k in ss_only_with_data["class"].values
]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()

if len(ss_only_no_data) > 0:
    print(f"\n=== {len(ss_only_no_data)} soundscape-only species with ZERO labeled data ===")
    display(ss_only_no_data[["species", "common_name", "class"]])

### 3.2 Co-occurrence partners of soundscape-only species

For these species, understanding what they co-occur with tells us: (a) what sounds MixIT would need to separate them from, and (b) whether ecological priors (e.g., "if you hear frog X, frog Y is likely nearby") could help.

In [ ]:
# For each soundscape-only species, find what it co-occurs with
ss_only_species_set = set(ss_only_df[ss_only_df["in_soundscape_labels"]]["species"])

cooccurrence_data = []
for sp in sorted(ss_only_species_set):
    # Find all segments containing this species
    sp_to_find = sp  # bind loop variable for lambda
    segments_with_sp = ss_labels_df[
        ss_labels_df["species_list"].apply(lambda x, s=sp_to_find: s in x)
    ]
    # Count co-occurring species
    partner_counts = Counter()
    for _, row in segments_with_sp.iterrows():
        for partner in row["species_list"]:
            if partner != sp:
                partner_counts[partner] += 1

    top_partners = partner_counts.most_common(5)
    partner_str = ", ".join(
        f"{tax_lookup.get(p, {}).get('common_name', p)[:20]} ({c})" for p, c in top_partners
    )
    cooccurrence_data.append(
        {
            "species": sp,
            "common_name": tax_lookup.get(sp, {}).get("common_name", "?"),
            "class": tax_lookup.get(sp, {}).get("class_name", "?"),
            "segments": len(segments_with_sp),
            "unique_partners": len(partner_counts),
            "top_5_partners": partner_str,
        }
    )

cooccurrence_df = pd.DataFrame(cooccurrence_data).sort_values("segments", ascending=False)
display(cooccurrence_df.head(15))

# How many co-occur with ONLY other soundscape-only species?
print("\n=== Average co-occurrence stats for soundscape-only species ===")
print(f"  Mean unique partners: {cooccurrence_df['unique_partners'].mean():.1f}")
print(f"  Mean segments: {cooccurrence_df['segments'].mean():.1f}")

---

## 4. Secondary Labels Analysis

12.3% of focal recordings in train_audio include `secondary_labels` — background species identified by the recordist. This is a source of multi-label supervision from focal recordings, relevant for both training and understanding the "missing labels" problem that MixIT addresses.

### 4.1 Secondary label coverage

In [ ]:
# Parse secondary labels
train_df["sec_labels_parsed"] = train_df["secondary_labels"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x != "[]" else []
)
train_df["has_secondary"] = train_df["sec_labels_parsed"].apply(len) > 0
train_df["n_secondary"] = train_df["sec_labels_parsed"].apply(len)

has_sec = train_df["has_secondary"].sum()
print(f"Recordings with secondary labels: {has_sec:,} ({100*has_sec/len(train_df):.1f}%)")
print(f"Recordings without: {len(train_df) - has_sec:,}")

# Distribution of secondary label counts
sec_counts = train_df[train_df["has_secondary"]]["n_secondary"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
sec_counts.plot.bar(ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Number of secondary labels")
ax.set_ylabel("Number of recordings")
ax.set_title("Distribution of Secondary Label Counts (among recordings that have them)")
for i, (_idx, val) in enumerate(sec_counts.items()):
    ax.text(i, val + 10, str(val), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

# By collection
sec_by_coll = train_df.groupby("collection")["has_secondary"].mean() * 100
print("\n=== % with secondary labels by collection ===")
for coll, pct in sec_by_coll.items():
    print(f"  {coll}: {pct:.1f}%")

### 4.2 Most common background species

In [ ]:
# Which species appear most frequently as secondary labels?
all_secondary = []
for labels in train_df["sec_labels_parsed"]:
    all_secondary.extend(labels)

sec_species_counts = Counter(all_secondary)
sec_species_df = pd.DataFrame(
    [
        (
            sp,
            count,
            tax_lookup.get(sp, {}).get("common_name", sp),
            tax_lookup.get(sp, {}).get("class_name", "?"),
        )
        for sp, count in sec_species_counts.most_common(30)
    ],
    columns=["species", "as_secondary", "common_name", "class"],
)

# Add primary count for comparison
sec_species_df["as_primary"] = sec_species_df["species"].map(lambda s: species_counts.get(s, 0))
sec_species_df["in_submission"] = sec_species_df["species"].isin(submission_species)

print("=== Top 30 background species ===")
display(sec_species_df)

# How many unique species appear as secondaries?
unique_sec_species = set(all_secondary)
in_submission_sec = unique_sec_species & submission_species
not_in_submission_sec = unique_sec_species - submission_species

print(f"\nUnique species appearing as secondary: {len(unique_sec_species)}")
print(f"  Of those, in submission set: {len(in_submission_sec)}")
print(f"  Of those, NOT in submission: {len(not_in_submission_sec)}")
if not_in_submission_sec:
    print(f"  Not in submission: {sorted(not_in_submission_sec)[:10]}...")

---

## Future Work

The following EDA areas were scoped but **not covered** in this notebook. They are lower priority for immediate model design but would be valuable for later optimization.

### 5. Audio Characteristics
- **Duration distributions** — how long are focal recordings per class/collection? Are iNat recordings systematically shorter?
- **Quality ratings** — XC provides 1–5 ratings; iNat has none. What's the quality distribution? Should we filter low-quality recordings?
- **Format/encoding** — verify sample rates, channels, codec consistency across the dataset

### 6. Geographic & Temporal Patterns
- **Recording site distribution** — 23 sites in train_soundscapes, but 4 sites hold 95% of data. How does species composition vary by site?
- **Time-of-day patterns** — soundscape filenames encode UTC time. Which species are nocturnal vs diurnal? Can we build time-of-day priors?
- **Seasonal variation** — do species assemblages shift across recording dates?
- **Site × species matrix** — which species are site-specific vs ubiquitous? Useful for test-time priors if test site IDs are available